# HW6: Hybrid CMA-ES (μ+λ) - Competition Notebook

**Course:** MCS-5993 Evolutionary Computation and Deep Learning  
**Assignment:** HW6 - Algorithm Competition  
**Student:** Harsha Yellela  

## Algorithm Features

- **(μ+λ) Population Strategy**: Maintains μ parents and generates λ offspring
- **Global Step Size (σ_g)**: Adapted using 1/5 success rule
- **Local Step Sizes (σ_local)**: Per-dimension adaptation from covariance
- **Multiple Restarts**: 2-3 restarts with adaptive parameters
- **Dimension-Aware**: Optimized for high-dimensional problems (dim=50)

## Competition Parameters (mu < 20, lambda < 20)

For dim=50: mu=8, lambda=18 (both < 20 as required)


## Part 1: HW6 Algorithm Implementation


In [1]:
import numpy as np

# global RNG for reproducibility
rng = np.random.default_rng(42)

def set_seed(seed: int = 42):
    """reset RNG seed"""
    global rng
    rng = np.random.default_rng(seed)


In [2]:
# helper functions

def clip_to_bounds(x: np.ndarray, bounds: np.ndarray) -> np.ndarray:
    """clip solution to bounds"""
    return np.clip(x, bounds[:, 0], bounds[:, 1])


def init_population(mu: int, bounds: np.ndarray) -> np.ndarray:
    """initialize population of mu parents randomly within bounds"""
    dim = bounds.shape[0]
    low = bounds[:, 0]
    high = bounds[:, 1]
    pop = rng.uniform(low=low, high=high, size=(mu, dim))
    return pop


def evaluate_population(pop: np.ndarray, objective) -> np.ndarray:
    """evaluate all individuals in population"""
    return np.array([objective(ind) for ind in pop])


In [3]:
# step-size adaptation functions

def init_sigmas(bounds: np.ndarray, sigma_global_scale: float = 0.3):
    """initialize global and local step sizes"""
    dim = bounds.shape[0]
    ranges = bounds[:, 1] - bounds[:, 0]
    ranges = np.where(ranges == 0, 1.0, ranges)  # avoid zero range
    
    # for high dimensions, slightly reduce initial step size for better convergence
    if dim >= 50:
        sigma_global_scale = 0.25  # smaller initial step for high dim
    elif dim >= 20:
        sigma_global_scale = 0.28
    
    # global sigma from average range
    sigma_g = sigma_global_scale * np.mean(ranges)
    
    # local sigmas from each dimension's range
    # scale down slightly for high dimensions
    local_scale = 0.25 if dim >= 50 else 0.3
    sigma_local = local_scale * ranges
    
    return sigma_g, sigma_local


def update_local_sigmas_from_parents(parents: np.ndarray,
                                     sigma_local: np.ndarray,
                                     alpha_corr: float = 0.5,
                                     min_sigma: float = 1e-8) -> np.ndarray:
    """update local step sizes using covariance matrix of parents"""
    mu, dim = parents.shape
    
    if mu < 2:
        return sigma_local
    
    # center parents
    centered = parents - np.mean(parents, axis=0, keepdims=True)
    
    # compute covariance matrix
    cov = np.cov(centered.T)
    cov = (cov + cov.T) / 2.0  # ensure symmetry
    
    # get variance per dimension
    var = np.diag(cov)
    var = np.maximum(var, 0.0)
    
    # base sigma from variance
    base_sigma = np.sqrt(var + 1e-12)
    
    # build correlation matrix
    std = np.sqrt(var + 1e-12)
    denom = np.outer(std, std)
    denom = np.where(denom == 0.0, 1e-12, denom)
    corr = cov / denom
    corr = np.clip(corr, -1.0, 1.0)
    
    # average correlation per dimension
    avg_abs_corr = np.mean(np.abs(corr), axis=1)
    
    # scale by correlation
    new_sigma_local = base_sigma * (1.0 + alpha_corr * avg_abs_corr)
    
    # smooth update
    beta = 0.5
    sigma_local = (1.0 - beta) * sigma_local + beta * new_sigma_local
    
    # minimum to avoid freezing
    sigma_local = np.maximum(sigma_local, min_sigma)
    
    return sigma_local


def update_global_sigma(sigma_g: float,
                        success_rate: float,
                        target_success: float = 0.2,
                        a_inc: float = 1.2,
                        b_dec: float = 0.85,
                        min_sigma: float = 1e-8,
                        max_sigma: float = 1e6) -> float:
    """update global step size using 1/5 rule"""
    if success_rate > target_success:
        sigma_g *= a_inc  # increase
    elif success_rate < target_success:
        sigma_g *= b_dec  # decrease
    
    sigma_g = max(min_sigma, min(max_sigma, sigma_g))
    return sigma_g


In [4]:
# main HW6 algorithm

def hw6_hybrid_cma_es(objective,
                      bounds: np.ndarray,
                      dim: int,
                      n_generations: int = 200,
                      mu: int = 4,
                      lam: int = 10,
                      seed: int | None = None):
    """main HW6 hybrid CMA-ES algorithm"""
    if seed is not None:
        set_seed(seed)
    
    bounds = np.asarray(bounds, dtype=float)
    assert bounds.shape == (dim, 2)
    
    # initialize parents
    parents = init_population(mu, bounds)
    parent_f = evaluate_population(parents, objective)
    
    # initialize step sizes
    sigma_g, sigma_local = init_sigmas(bounds)
    
    # track best
    best_idx = np.argmin(parent_f)
    best_f = float(parent_f[best_idx])
    best_x = parents[best_idx].copy()
    
    history = [best_f]
    
    for gen in range(n_generations):
        # generate offspring
        offspring = np.empty((lam, dim))
        
        for i in range(lam):
            # pick random parent
            p_idx = rng.integers(0, mu)
            parent = parents[p_idx]
            
            # per-dimension step sizes
            step_sizes = sigma_g * sigma_local
            
            # mutate
            step = rng.normal(loc=0.0, scale=step_sizes, size=dim)
            child = parent + step
            child = clip_to_bounds(child, bounds)
            offspring[i] = child
        
        # evaluate offspring
        offspring_f = evaluate_population(offspring, objective)
        
        # compute success rate for 1/5 rule
        parent_baseline = np.median(parent_f)
        successes = np.sum(offspring_f < parent_baseline)
        success_rate = successes / max(1, lam)
        
        # update global sigma
        sigma_g = update_global_sigma(sigma_g, success_rate)
        
        # (μ+λ) selection
        combined = np.vstack([parents, offspring])
        combined_f = np.concatenate([parent_f, offspring_f])
        
        # sort by fitness
        idx = np.argsort(combined_f)
        combined = combined[idx]
        combined_f = combined_f[idx]
        
        # keep best μ
        parents = combined[:mu]
        parent_f = combined_f[:mu]
        
        # update local sigmas
        sigma_local = update_local_sigmas_from_parents(parents, sigma_local)
        
        # update best
        if parent_f[0] < best_f:
            best_f = float(parent_f[0])
            best_x = parents[0].copy()
        
        history.append(best_f)
    
    return best_f, best_x, history


In [5]:
# competition wrapper with multiple restarts

def HW6(f, bounds, dim, n_generations=200, seed=None, **kwargs):
    """competition wrapper with multiple restarts"""
    bounds = np.asarray(bounds, dtype=float)
    
    # adapt parameters based on dimension and search space size
    range_size = np.mean(bounds[:, 1] - bounds[:, 0])
    
    # for high dimensions, use larger populations (but < 20 constraint)
    if dim >= 50:
        # high dimension: need more exploration (all values < 20 as required)
        n_restarts = 3
        mu_val = 8  # < 20
        lam_val = 18  # < 20 (conservative limit)
        gen_per_run = n_generations if n_generations != 200 else 350
    elif dim >= 20:
        # medium-high dimension (all values < 20)
        n_restarts = 3
        mu_val = 7  # < 20
        lam_val = 16  # < 20
        gen_per_run = n_generations if n_generations != 200 else 330
    elif range_size > 100:  # large space
        n_restarts = 3
        gen_per_run = 350
        mu_val = 6
        lam_val = 15
    elif range_size > 10:  # medium
        n_restarts = 2
        gen_per_run = 300
        mu_val = 5
        lam_val = 12
    else:  # small
        n_restarts = 2
        gen_per_run = 250
        mu_val = 4
        lam_val = 10
    
    if n_generations != 200:
        gen_per_run = n_generations
        if n_generations < 150:
            n_restarts = max(1, n_restarts - 1)
    
    best_overall = float('inf')
    
    # multiple restarts
    for restart in range(n_restarts):
        restart_seed = (seed + restart * 10000) if seed is not None else restart * 10000
        
        try:
            best_f, best_x, history = hw6_hybrid_cma_es(
                objective=f,
                bounds=bounds,
                dim=dim,
                n_generations=gen_per_run,
                mu=mu_val,
                lam=lam_val,
                seed=restart_seed
            )
            
            if best_f < best_overall:
                best_overall = best_f
        
        except Exception as e:
            print(f"Warning: Restart {restart} failed: {e}", flush=True)
            continue
    
    # fallback if all failed
    if best_overall == float('inf'):
        best_f, _, _ = hw6_hybrid_cma_es(
            objective=f,
            bounds=bounds,
            dim=dim,
            n_generations=n_generations,
            mu=4,
            lam=10,
            seed=seed
        )
        return best_f
    
    return best_overall


## Part 2: Benchmark Functions


In [6]:
# 10 benchmark functions

def sphere(x):
    """Sphere function - unimodal"""
    return np.sum(x**2)

def dixon_price(x):
    """Dixon-Price function - Competition function"""
    x = np.array(x)
    n = len(x)
    
    # first term (x1 - 1)^2
    term1 = (x[0] - 1)**2
    
    # loop part, vectorized
    # Formula: sum( i * (2*x_i^2 - x_{i-1})^2 ) for i from 2 to n
    indices = np.arange(2, n + 1)
    term2 = np.sum(indices * (2 * x[1:]**2 - x[:-1])**2)
    
    return term1 + term2

def rosenbrock(x):
    """Rosenbrock function - unimodal valley"""
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def rastrigin(x):
    """Rastrigin function - highly multimodal"""
    A = 10
    n = len(x)
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x))

def ackley(x):
    """Ackley function - multimodal"""
    n = len(x)
    sum_sq = np.sum(x**2)
    sum_cos = np.sum(np.cos(2 * np.pi * x))
    return -20 * np.exp(-0.2 * np.sqrt(sum_sq / n)) - np.exp(sum_cos / n) + 20 + np.e

def griewank(x):
    """Griewank function - multimodal"""
    sum_sq = np.sum(x**2)
    prod_cos = np.prod(np.cos(x / np.sqrt(np.arange(1, len(x) + 1))))
    return 1 + sum_sq / 4000 - prod_cos

def schwefel(x):
    """Schwefel function - multimodal"""
    n = len(x)
    return 418.9829 * n - np.sum(x * np.sin(np.sqrt(np.abs(x))))

def levy(x):
    """Levy function - multimodal"""
    w = 1 + (x - 1) / 4
    term1 = np.sin(np.pi * w[0])**2
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * w[:-1] + 1)**2))
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2 * np.pi * w[-1])**2)
    return term1 + term2 + term3

def zakharov(x):
    """Zakharov function - unimodal"""
    sum1 = np.sum(x**2)
    sum2 = np.sum(0.5 * np.arange(1, len(x) + 1) * x)
    return sum1 + sum2**2 + sum2**4

def lunacek_bi_rastrigin(x):
    """Lunacek Bi-Rastrigin - simplified version"""
    return rastrigin(x) + np.sum((x - 2.5)**2)

def hybrid_composition(x):
    """Hybrid Composition - combination of multiple functions"""
    return 0.3 * sphere(x) + 0.3 * rastrigin(x) + 0.4 * griewank(x)


## Part 3: Baseline Algorithms


In [7]:
# baseline algorithms for comparison

def random_search(f, bounds, dim, n_generations=200, seed=None):
    """Random Search baseline"""
    if seed is not None:
        np.random.seed(seed)
    
    best_f = float('inf')
    best_x = None
    
    n_evals = n_generations * 10
    for _ in range(n_evals):
        x = np.random.uniform(bounds[:, 0], bounds[:, 1])
        fx = f(x)
        if fx < best_f:
            best_f = fx
            best_x = x
    
    return best_f


def one_fifth_rule_es(f, bounds, dim, n_generations=200, seed=None):
    """1/5 Rule Evolution Strategy"""
    if seed is not None:
        np.random.seed(seed)
    
    # initialize
    x = np.random.uniform(bounds[:, 0], bounds[:, 1])
    sigma = 0.3 * np.mean(bounds[:, 1] - bounds[:, 0])
    
    best_f = f(x)
    best_x = x.copy()
    
    lam = 10
    for gen in range(n_generations):
        # generate offspring
        successes = 0
        for _ in range(lam):
            x_new = x + sigma * np.random.randn(dim)
            x_new = np.clip(x_new, bounds[:, 0], bounds[:, 1])
            f_new = f(x_new)
            
            if f_new < f(x):
                x = x_new
                successes += 1
                if f_new < best_f:
                    best_f = f_new
                    best_x = x_new.copy()
        
        # adapt sigma using 1/5 rule
        success_rate = successes / lam
        if success_rate > 0.2:
            sigma *= 1.2  # increase
        elif success_rate < 0.2:
            sigma *= 0.85  # decrease
    
    return best_f


def mu_plus_lambda_es(f, bounds, dim, n_generations=200, seed=None):
    """(μ+λ) Evolution Strategy"""
    if seed is not None:
        np.random.seed(seed)
    
    mu = 4
    lam = 10
    
    # initialize population
    population = [np.random.uniform(bounds[:, 0], bounds[:, 1]) for _ in range(mu)]
    fitness = [f(ind) for ind in population]
    
    best_f = min(fitness)
    best_x = population[fitness.index(best_f)].copy()
    
    for gen in range(n_generations):
        # generate offspring
        offspring = []
        for _ in range(lam):
            parent = population[np.random.randint(mu)]
            sigma = 0.3 * np.mean(bounds[:, 1] - bounds[:, 0])
            child = parent + sigma * np.random.randn(dim)
            child = np.clip(child, bounds[:, 0], bounds[:, 1])
            offspring.append(child)
        
        # evaluate offspring
        offspring_fitness = [f(ind) for ind in offspring]
        
        # select best μ from parents + offspring
        combined = population + offspring
        combined_fitness = fitness + offspring_fitness
        
        indices = np.argsort(combined_fitness)[:mu]
        population = [combined[i] for i in indices]
        fitness = [combined_fitness[i] for i in indices]
        
        if fitness[0] < best_f:
            best_f = fitness[0]
            best_x = population[0].copy()
    
    return best_f


def mu_plus_lambda_es_variant(f, bounds, dim, n_generations=200, seed=None):
    """(μ+λ) ES variant"""
    return mu_plus_lambda_es(f, bounds, dim, n_generations, seed)


In [8]:
# algorithm dictionary

algorithms = {
    'Random Search': random_search,
    '1/5 Rule ES': one_fifth_rule_es,
    '(μ+λ)-ES': mu_plus_lambda_es,
    '(μ+λ)-ES Vari': mu_plus_lambda_es_variant,
    'HW6': HW6,
}


In [9]:
# comparison functions

def compare_algorithms(algorithms, test_functions, n_runs=10, n_generations=200):
    """Compare all algorithms on all test functions"""
    results = {}
    
    for func_name, func_info in test_functions.items():
        print(f"\\nTesting on {func_name}...")
        results[func_name] = {}
        
        for alg_name, alg_func in algorithms.items():
            print(f"  Running {alg_name}...", end=' ')
            
            # run multiple times for average
            runs = []
            for run in range(n_runs):
                try:
                    best_f = alg_func(
                        f=func_info['function'],
                        bounds=func_info['bounds'],
                        dim=func_info['dim'],
                        n_generations=n_generations,
                        seed=run
                    )
                    runs.append(best_f)
                except Exception as e:
                    print(f"Error: {e}")
                    runs.append(float('inf'))
            
            results[func_name][alg_name] = np.mean(runs)
            print(f"Mean: {results[func_name][alg_name]:.6e}")
    
    return results


def print_comparison_table(results, dim=2):
    """Print comparison results in formatted table"""
    print("\\n" + "="*120)
    print(f"{f'D={dim}':<20} ALGORITHM COMPARISON SUMMARY")
    print("="*120)
    
    # header
    header = f"{'Function':<20}"
    for alg_name in algorithms.keys():
        header += f"{alg_name:<20}"
    header += f"{'Winner':<20}"
    print(header)
    print("-"*120)
    
    # count wins
    wins = {alg: 0 for alg in algorithms.keys()}
    
    # print results for each function
    for func_name, func_results in results.items():
        row = f"{func_name:<20}"
        
        # find best result
        best_result = min(func_results.values())
        winner = [alg for alg, res in func_results.items() if res == best_result][0]
        wins[winner] += 1
        
        # print all algorithm results
        for alg_name in algorithms.keys():
            result = func_results[alg_name]
            row += f"{result:<20.6e}"
        
        row += f"{winner:<20}"
        print(row)
    
    # print summary
    print("-"*120)
    row = f"{'TOTAL WINS':<20}"
    for alg_name in algorithms.keys():
        row += f"{wins[alg_name]:<20}"
    print(row)
    
    # overall winner
    overall_winner = max(wins.items(), key=lambda x: x[1])
    print(f"\\n{'OVERALL WINNER':<20}{overall_winner[0]:<20}")
    print("="*120)


In [10]:
def main(dim=50, max_evaluations=5000, runs=5):
    """competition main function"""
    # convert max_evaluations to n_generations
    # using average mu=5, lam=12 for estimation
    avg_pop_size = 17  # mu + lambda average
    n_generations = max(1, max_evaluations // avg_pop_size)
    
    # create all 10 functions with specified dimension bounds
    competition_functions = {
        'Dixon_Price': {
            'function': dixon_price,
            'bounds': np.array([[-10, 10]] * dim),
            'dim': dim,
            'global_min': 1e-5
        },
        'Rosenbrock': {
            'function': rosenbrock,
            'bounds': np.array([[-2.048, 2.048]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Rastrigin': {
            'function': rastrigin,
            'bounds': np.array([[-5.12, 5.12]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Ackley': {
            'function': ackley,
            'bounds': np.array([[-32.768, 32.768]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Griewank': {
            'function': griewank,
            'bounds': np.array([[-600, 600]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Schwefel': {
            'function': schwefel,
            'bounds': np.array([[-500, 500]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Lunacek BiRstrgn': {
            'function': lunacek_bi_rastrigin,
            'bounds': np.array([[-5.12, 5.12]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Levy': {
            'function': levy,
            'bounds': np.array([[-10, 10]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Zakharov': {
            'function': zakharov,
            'bounds': np.array([[-5, 10]] * dim),
            'dim': dim,
            'global_min': 0.0
        },
        'Hybrid Composition': {
            'function': hybrid_composition,
            'bounds': np.array([[-5, 5]] * dim),
            'dim': dim,
            'global_min': 0.0
        }
    }
    
    print(f"Running Competition: dim={dim}, max_evaluations={max_evaluations}, runs={runs}")
    print(f"Using {n_generations} generations (estimated from {max_evaluations} max evaluations)")
    
    # run comparison
    results = compare_algorithms(
        algorithms=algorithms,
        test_functions=competition_functions,
        n_runs=runs,
        n_generations=n_generations
    )
    
    # print formatted table with dimension
    print_comparison_table(results, dim=dim)
    
    print("\\n✅ Competition comparison complete!")
    return results


In [11]:
# Run competition
results = main(dim=50, max_evaluations=5000, runs=5)


Running Competition: dim=50, max_evaluations=5000, runs=5
Using 294 generations (estimated from 5000 max evaluations)
\nTesting on Dixon_Price...
  Running Random Search... Mean: 3.857354e+06
  Running 1/5 Rule ES... Mean: 1.149679e+01
  Running (μ+λ)-ES... Mean: 6.840338e+06
  Running (μ+λ)-ES Vari... Mean: 6.840338e+06
  Running HW6... Mean: 1.556806e+04
\nTesting on Rosenbrock...
  Running Random Search... Mean: 9.312474e+03
  Running 1/5 Rule ES... Mean: 8.175148e+01
  Running (μ+λ)-ES... Mean: 1.543882e+04
  Running (μ+λ)-ES Vari... Mean: 1.543882e+04
  Running HW6... Mean: 2.397299e+02
\nTesting on Rastrigin...
  Running Random Search... Mean: 6.704317e+02
  Running 1/5 Rule ES... Mean: 3.786789e+02
  Running (μ+λ)-ES... Mean: 7.471911e+02
  Running (μ+λ)-ES Vari... Mean: 7.471911e+02
  Running HW6... Mean: 2.740305e+02
\nTesting on Ackley...
  Running Random Search... Mean: 2.050886e+01
  Running 1/5 Rule ES... Mean: 1.929874e+01
  Running (μ+λ)-ES... Mean: 2.094037e+01
  Runnin